# Boosting: AdaBoost and gradient boosting

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Boosting builds **weak** models (a bit better than chance, such as one-split trees or "stumps") **one after another**; each one focuses on the previous mistakes. Their weighted sum becomes a **strong** model.

## AdaBoost step by step
- Every sample starts with the same weight $w_i=1/n$.
- For $m=1..M$:
- Train a stump with the weights; compute its weighted error $\varepsilon_m$.
- Model weight: $\alpha_m=\frac12\ln\frac{1-\varepsilon_m}{\varepsilon_m}$ (lower error = bigger vote).
- Increase the weights of **wrong** samples and decrease the right ones: $w_i\leftarrow w_i e^{-\alpha_m y_ih_m(x_i)}$, then normalize.
- Final prediction: $\text{sign}\big(\sum_m\alpha_mh_m(x)\big)$.

## A worked round
10 samples; the first stump gets 3 wrong: $\varepsilon=0.3$ and $\alpha=\frac12\ln\frac{0.7}{0.3}=0.42$.

| Samples | Weight before | Factor | Weight after (normalized) |
|---|---|---|---|
| 7 correct | 0.1 | $e^{-0.42}=0.65$ | 0.071 |
| 3 wrong | 0.1 | $e^{0.42}=1.53$ | 0.167 |

The 3 wrong samples now hold **half** of the total weight, so the next stump has to get them right.



In [ ]:
import numpy as np

# One AdaBoost round by hand: 10 samples, 3 mistakes
w = np.full(10, 0.1)
wrong = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0], bool)
eps = w[wrong].sum()
alpha = 0.5 * np.log((1 - eps) / eps)
w = w * np.exp(np.where(wrong, alpha, -alpha))
w /= w.sum()
print("error =", eps, " alpha =", round(alpha, 3))
print("new weights:", w.round(3), " weight of wrong samples:", w[wrong].sum().round(3))



> [AdaBoost with 1, 5 and 100 stumps on circles. Simple axis-parallel cuts together build a circle.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/boost_cls.png)

*AdaBoost with 1, 5 and 100 stumps on circles. Simple axis-parallel cuts together build a circle.*

[[lab:cls_ada]]

## Gradient boosting for classification
As in the regression lesson, but the trees learn the **gradient of the log loss** ($y-p$) and the output goes through a sigmoid to become a probability. `HistGradientBoostingClassifier` is scikit-learn's fastest option, and XGBoost, LightGBM and CatBoost are its famous versions.



In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)
for name, m in [("AdaBoost 200", AdaBoostClassifier(n_estimators=200, random_state=0)),
                ("GradientBoosting", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=2, random_state=0)),
                ("HistGradientBoosting", HistGradientBoostingClassifier(random_state=0))]:
    print(f"{name:22s} accuracy = {cross_val_score(m, X, y, cv=5).mean():.3f}")



| | AdaBoost | Gradient boosting |
|---|---|---|
| How it focuses | Reweights wrong samples | Trains on the gradient of the loss |
| Loss | Exponential | Any differentiable loss |
| Sensitivity to noise and outliers | High | Lower |
| Use today | Teaching, small data | The standard for tabular data |

### Exercise

Use staged_score to plot AdaBoost's test accuracy after each stump on breast cancer.

In [ ]:
# Your answer here